In [1]:
# =============================================================================
# IMPORTS Y CONFIGURACIÓN
# =============================================================================
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import json
import gc
import psutil
import os
from pathlib import Path
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.decomposition import PCA
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuración para plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 7)

# Paths
RAW_DIR = Path("../data/raw/cluvi/")
OUTPUT_DIR = "../data/process/"  # Adaptado si es necesario
REPORT_DIR = Path("../reports/eda_cluvi/")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("EDA COMPLETO Y MEJORADO - DATASET CLUVI (MENÚS Y CATEGORÍAS)")
print("="*80)

EDA COMPLETO Y MEJORADO - DATASET CLUVI (MENÚS Y CATEGORÍAS)


In [2]:
# =============================================================================
# FUNCIONES AUXILIARES
# =============================================================================

def save_report(data, filename, format='json'):
    """Guarda reportes en formato JSON o CSV"""
    filepath = REPORT_DIR / filename
    if format == 'json':
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    elif format == 'csv' and isinstance(data, (pl.DataFrame, pd.DataFrame)):
        if isinstance(data, pl.DataFrame):
            data.write_csv(filepath)
        else:
            data.to_csv(filepath, index=False)
    print(f"✓ Reporte guardado: {filepath}")

def plot_and_save(fig, filename):
    """Guarda figura y la muestra"""
    filepath = REPORT_DIR / filename
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"✓ Gráfico guardado: {filepath}")
    plt.show()

def calculate_gini(counts):
    """Calcula el coeficiente de Gini para medir desbalance"""
    sorted_counts = np.sort(counts)
    n = len(counts)
    cumsum = np.cumsum(sorted_counts)
    return (2 * np.sum((np.arange(1, n+1)) * sorted_counts)) / (n * cumsum[-1]) - (n + 1) / n

def print_section(title):
    """Imprime encabezado de sección"""
    print("\n" + "="*80)
    print(f"{title}")
    print("="*80)

def liberar_ram():
    """Libera RAM y muestra uso actual"""
    process = psutil.Process(os.getpid())
    mem_antes = process.memory_info().rss / 1024 / 1024  # MB
    
    # Colectar garbage
    gc.collect()
    
    mem_despues = process.memory_info().rss / 1024 / 1024
    print(f"RAM liberada: {mem_antes - mem_despues:.2f} MB")
    print(f"RAM actual: {mem_despues:.2f} MB")

In [7]:
# =============================================================================
# CARGA DE DATOS (concatenando todos los CSVs)
# =============================================================================
print_section("CARGA DE DATOS (Todos los CSVs en ../data/raw/CLuvi/)")

import glob
import os

print(f"pwd {os.getcwd()}")

csv_files = sorted(glob.glob(str(RAW_DIR / "*.csv")))
print(f"Intentando cargar archivos CSV desde: {RAW_DIR}")
if not csv_files:
    print(f"✗ No se encontraron archivos CSV en {RAW_DIR}")
    # Diagnóstico: ¿existe el archivo específico?
    test_file = RAW_DIR / "dimproducts_202509300127.csv"
    if test_file.exists():
        print(f"¡El archivo {test_file} SÍ existe! Pero glob no lo encontró. Revisa mayúsculas/minúsculas en la ruta o extensión.")
    else:
        print(f"El archivo {test_file} NO existe según os.path.exists().")
    print("Por favor, verifica que la ruta y los archivos existan antes de continuar.")
    # Salida temprana: no hay datos, DataFrame vacío para evitar errores posteriores
    df = pl.DataFrame()
else:
    print(f"Archivos CSV encontrados: {len(csv_files)}")
    for f in csv_files:
        print(f"  - {f}")

    # Leer y concatenar todos los CSVs encontrados
    dfs = []
    for f in csv_files:
        try:
            df_tmp = pl.read_csv(f, separator=";", infer_schema_length=10000)
            dfs.append(df_tmp)
        except Exception as e:
            print(f"Error al leer {f}: {e}")

    if not dfs:
        print("✗ No se pudo cargar ningún archivo CSV.")
        df = pl.DataFrame()
    else:
        df_full = pl.concat(dfs, how="vertical", rechunk=True)
        print(f"✓ DataFrame concatenado: {df_full.shape}")

        # Seleccionar columnas relevantes y limpiar
        df = df_full.select([
            pl.col("label").str.to_lowercase().alias("label"),
            pl.col("standard_label").str.to_lowercase().alias("standard_label"),
            pl.col("description").str.to_lowercase().alias("description"),
            pl.col("category").str.to_lowercase().alias("category"),
            pl.col("main_category").str.to_lowercase().alias("main_category")
        ])

        # Filtrar filas con label no nulo (asumiendo es clave primaria)
        df = df.filter(pl.col("label").is_not_null())

        # Agregar columna menu_id como string tipo MEN_00VALOR
        df = df.with_row_count(name="menu_id_num", offset=1)
        df = df.with_columns(
            (pl.lit("MEN_") + pl.col("menu_id_num").cast(pl.Utf8).str.zfill(6)).alias("menu_id")
        ).drop("menu_id_num")

        print(f"✓ DataFrame final: {df.shape}")
        print(f"Columnas: {df.columns}")

        # Sample si es muy grande (1/2.5 para consistencia)
        if df.height > 10000:
            df = df.sample(n=int(df.height / 2.5), seed=42)
            print(f"✓ Muestra (1/2.5): {df.shape}")


CARGA DE DATOS (Todos los CSVs en ../data/raw/CLuvi/)
pwd /home/daniel-linux/Tesis/Recolectar_Datos/notebooks
Intentando cargar archivos CSV desde: ../data/raw/cluvi
Archivos CSV encontrados: 1
  - ../data/raw/cluvi/dimproducts_202509300127.csv
✓ DataFrame concatenado: (1110178, 5)
✓ DataFrame final: (1110178, 6)
Columnas: ['label', 'standard_label', 'description', 'category', 'main_category', 'menu_id']
✓ Muestra (1/2.5): (444071, 6)


In [ ]:
# Guardar el DataFrame limpio a un nuevo archivo CSV para uso posterior
output_path = "../data/cleaned/menu_cleaned.csv"
try:
    df.write_csv(output_path, separator=";")
    print(f"✓ Datos limpios guardados en: {output_path}")
except Exception as e:
    print(f"Error al guardar los datos limpios: {e}")


✓ Datos limpios guardados en: ../data/cleaned/menu_cleaned.csv


: 

In [ ]:
# =============================================================================
# FASE 2: PREPROCESAMIENTO TEXTUAL Y AJUSTE ESTRUCTURAL
# =============================================================================
print_section("FASE 2: PREPROCESAMIENTO TEXTUAL")

sentences_with_annots = main_clean.group_by("sentence_id").agg([
    pl.count().alias("num_entities"),
    pl.col("entity_length_chars").mean().alias("avg_entity_length")
]).join(sentences_df, on="sentence_id", how="left")

# 1. Manejo de Oraciones Extensas (truncar >1000 chars; o eliminar top 1%)
long_threshold = 1000
long_sentences = sentences_with_annots.filter(pl.col("sentence_length_chars") > long_threshold)
print(f"Oraciones largas truncadas/eliminadas: {len(long_sentences)}")
# Truncar sentences (agregar columna truncada)
sentences_df = sentences_df.with_columns([
    pl.when(pl.col("sentence_length_chars") > long_threshold)
    .then(pl.col("sentence").str.slice(0, long_threshold))
    .otherwise(pl.col("sentence"))
    .alias("sentence_truncated")
]).with_columns(pl.col("sentence_truncated").str.len_chars().alias("sentence_length_trunc"))

# 2. Filtrado de URLs/Entidades Largas (detectar URLs y eliminar)
url_pattern = r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+"
long_urls = main_clean.filter(
    (pl.col("entity_length_chars") > 50) &  # Umbral aproximado
    pl.col("entity").str.contains(url_pattern)
)
print(f"URLs/Entidades largas filtradas: {len(long_urls)}")
main_clean = main_clean.filter(~pl.col("entity").str.contains(url_pattern))
phase2_stats = {"long_entities_filtered": len(long_urls)}

# 3. Normalización Textual (opcional: lower case para no-ALL CAPS)
main_clean = main_clean.with_columns([
    pl.when(pl.col("is_all_caps"))
    .then(pl.col("entity"))
    .otherwise(pl.col("entity").str.to_lowercase())
    .alias("entity_normalized")
])

# 4. Validar Correlaciones (calcular post-limpieza)
corr_cols = ["char_start", "char_end", "token_start", "token_end", "entity_length_chars", "entity_length_tokens"]
corr_df = main_clean.select(corr_cols).to_pandas().corr()
print("Correlaciones post-limpieza:")
print(corr_df["entity_length_chars"]["entity_length_tokens"])  # Debería ~0.83
phase2_stats["post_corr_chars_tokens"] = float(corr_df["entity_length_chars"]["entity_length_tokens"])

liberar_ram()
save_report(phase2_stats, 'phase2_stats.json')
print(f"✓ Post-Fase 2: {main_clean.shape}")


FASE 2: PREPROCESAMIENTO TEXTUAL


/tmp/ipykernel_77096/4235887224.py:7: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_entities"),


Oraciones largas truncadas/eliminadas: 413517
URLs/Entidades largas filtradas: 14
Correlaciones post-limpieza:
0.7970454733911962
RAM liberada: 0.00 MB | Actual: 15737.36 MB
✓ Reporte guardado: ../reports/cleaning/phase2_stats.json
✓ Post-Fase 2: (4035825, 22)


: 